[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Murua-Alvaro/urban/blob/feat/denue-colab-workspace/notebooks/denue/00_COLAB_START_HERE.ipynb)

# Urban · DENUE histórico · START HERE

Este es el notebook principal para ejecutar el pipeline DENUE de Urban en Google Colab.

**Uso normal:** abre este notebook en Colab y elige **Entorno de ejecución → Ejecutar todas**.

El notebook:
1. clona la rama correcta de GitHub;
2. instala el paquete `urban-denue`;
3. busca y verifica el ZIP auditado;
4. usa Google Drive como caché si lo autorizas;
5. si el ZIP aún no está en GitHub ni en Drive, te pide seleccionarlo una sola vez;
6. ejecuta directamente el pipeline Python (sin subprocesos ocultos);
7. valida 25 ediciones, totales y consistencia territorial;
8. genera EDA y, opcionalmente, análisis espacial, features y modelos;
9. termina con controles automáticos de integridad.

In [ ]:
#@title 1. Configuración
STAGE = "eda" #@param ["load", "validate", "clean", "eda", "spatial", "features", "model", "all"]
USE_DRIVE_CACHE = True #@param {type:"boolean"}
ALLOW_MANUAL_UPLOAD = True #@param {type:"boolean"}
WITH_MORAN = False #@param {type:"boolean"}
FORCE_REDOWNLOAD = False #@param {type:"boolean"}

REPO_URL = "https://github.com/Murua-Alvaro/urban.git"
BRANCH = "feat/denue-colab-workspace"
WORKDIR = "/content/urban"

print("Configuración")
print("  STAGE =", STAGE)
print("  USE_DRIVE_CACHE =", USE_DRIVE_CACHE)
print("  WITH_MORAN =", WITH_MORAN)


In [ ]:
#@title 2. Preparar el entorno
import os
import pathlib
import platform
import shutil
import subprocess
import sys

workdir = pathlib.Path(WORKDIR)
if workdir.exists():
    shutil.rmtree(workdir)

def run_checked(cmd, cwd=None):
    print("$", " ".join(map(str, cmd)))
    completed = subprocess.run(
        cmd,
        cwd=cwd,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    print(completed.stdout)
    if completed.returncode != 0:
        raise RuntimeError(
            f"Comando falló con código {completed.returncode}: {' '.join(map(str, cmd))}"
        )
    return completed

run_checked([
    "git", "clone", "--depth", "1", "--single-branch",
    "--branch", BRANCH, REPO_URL, WORKDIR
])

os.chdir(WORKDIR)

print("Python:", sys.version)
print("Plataforma:", platform.platform())
run_checked(["git", "rev-parse", "--short", "HEAD"], cwd=WORKDIR)

run_checked([
    sys.executable, "-m", "pip", "install", "-q",
    "-r", "requirements-colab.txt"
], cwd=WORKDIR)

run_checked([
    sys.executable, "-m", "pip", "install", "-q",
    "--no-deps", "-e", "."
], cwd=WORKDIR)

from urban_denue.pipeline import environment_snapshot
print("Entorno urban-denue listo:")
print(environment_snapshot())


In [ ]:
#@title 3. Definir caché y carpeta de trabajo
from pathlib import Path

drive_mounted = False

if USE_DRIVE_CACHE:
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        DENUE_ROOT = Path("/content/drive/MyDrive/Urban_DENUE_Cache")
        drive_mounted = True
        print("Caché persistente en Google Drive activada.")
    except Exception as exc:
        print("No se pudo montar Drive; usaré almacenamiento temporal de Colab.")
        print("Detalle:", repr(exc))
        DENUE_ROOT = Path("/content/urban-denue")
else:
    DENUE_ROOT = Path("/content/urban-denue")

DENUE_ROOT.mkdir(parents=True, exist_ok=True)

print("DENUE_ROOT =", DENUE_ROOT)
print("Persistente entre sesiones =", drive_mounted)


In [ ]:
#@title 4. Localizar y verificar el ZIP DENUE
import hashlib
import urllib.error

from urban_denue.archive import acquire_archive, verify_archive, sha256_file
from urban_denue.config import paths
from urban_denue.constants import (
    ARCHIVE_NAME,
    ARCHIVE_SHA256,
    ARCHIVE_SIZE,
    ARCHIVE_URL,
)

p = paths(DENUE_ROOT)
archive_path = p.archive
archive_path.parent.mkdir(parents=True, exist_ok=True)

archive_source = None

# 1) Caché local/Drive
if archive_path.exists():
    try:
        verify_archive(archive_path)
        archive_source = "cache"
        print("✓ ZIP válido encontrado en caché.")
    except Exception as exc:
        print("La copia existente no es válida; se eliminará.")
        print("Detalle:", repr(exc))
        archive_path.unlink(missing_ok=True)

# 2) GitHub
if archive_source is None:
    try:
        acquired = acquire_archive(
            archive_path,
            url=ARCHIVE_URL,
            force=FORCE_REDOWNLOAD,
        )
        verify_archive(acquired)
        archive_source = "github"
        print("✓ ZIP descargado y verificado desde GitHub.")
    except urllib.error.HTTPError as exc:
        if exc.code != 404:
            raise
        print("El ZIP todavía no está físicamente en GitHub (HTTP 404).")
    except Exception as exc:
        print("No fue posible obtener el ZIP automáticamente.")
        print("Detalle:", repr(exc))

# 3) Subida manual de respaldo
if archive_source is None:
    if not ALLOW_MANUAL_UPLOAD:
        raise FileNotFoundError(
            "No existe una copia válida del ZIP en caché ni en GitHub. "
            "Activa ALLOW_MANUAL_UPLOAD o sube el ZIP a la rama data/denue-historico."
        )

    try:
        from google.colab import files
    except ImportError as exc:
        raise RuntimeError(
            "La subida manual solo está disponible al ejecutar este notebook en Google Colab."
        ) from exc

    print("\nSelecciona el ZIP auditado:")
    print("  Growa_DENUE_Sinaloa_Historico_2010_2026_GitHub(1).zip")
    uploaded = files.upload()

    zip_names = [name for name in uploaded if name.lower().endswith(".zip")]
    if len(zip_names) != 1:
        raise ValueError(
            f"Debes seleccionar exactamente un ZIP; recibí: {list(uploaded)}"
        )

    uploaded_name = zip_names[0]
    archive_path.write_bytes(uploaded[uploaded_name])

    try:
        verify_archive(archive_path)
    except Exception:
        archive_path.unlink(missing_ok=True)
        raise

    archive_source = "manual_upload"
    print("✓ ZIP subido y verificado.")

# Verificación final independiente
verify_archive(archive_path)
actual_sha = sha256_file(archive_path)
actual_size = archive_path.stat().st_size

assert actual_sha == ARCHIVE_SHA256
assert actual_size == ARCHIVE_SIZE

print("\nArchivo DENUE listo")
print("  fuente:", archive_source)
print("  ruta:", archive_path)
print("  bytes:", f"{actual_size:,}")
print("  SHA-256:", actual_sha)


In [ ]:
#@title 5. Ejecutar pipeline
import json
import traceback

from urban_denue.pipeline import run_pipeline

VALID_STAGES = {"load", "validate", "clean", "eda", "spatial", "features", "model", "all"}
if STAGE not in VALID_STAGES:
    raise ValueError(f"STAGE inválido: {STAGE}")

pipeline_stage = "all" if STAGE in {"model", "all"} else STAGE

print("Ejecutando pipeline directamente en el kernel de Colab.")
print("Etapa solicitada:", STAGE)
print("Hasta etapa:", pipeline_stage)

try:
    manifest = run_pipeline(
        root=DENUE_ROOT,
        through=pipeline_stage,
        skip_moran=not WITH_MORAN,
        force_download=False,
    )
except Exception:
    print("\nERROR DEL PIPELINE — traceback completo:\n")
    traceback.print_exc()
    raise

print("\n✓ Pipeline base terminado correctamente.")
print(json.dumps(manifest["stages"], ensure_ascii=False, indent=2))


In [ ]:
#@title 6. Modelado (solo si STAGE = model o all)
import pandas as pd

model_outputs = None

if STAGE in {"model", "all"}:
    from urban_denue.config import paths
    from urban_denue.modeling import run_models, write_models

    p = paths(DENUE_ROOT)
    features_path = p.processed / "feature_store" / "grid_features.parquet"

    if not features_path.exists():
        raise FileNotFoundError(
            f"No existe el feature store requerido para modelado: {features_path}"
        )

    grid_features = pd.read_parquet(features_path)
    model_artifacts = run_models(grid_features)
    model_outputs = write_models(
        model_artifacts,
        p.processed / "models" / "grid_dynamics",
    )

    print("✓ Modelado terminado.")
    print("Observaciones:", model_artifacts.diagnostics.get("n"))
    print("Geografías:", model_artifacts.diagnostics.get("geographies"))
    print("Salidas:")
    for key, value in model_outputs.items():
        print(" ", key, "->", value)
else:
    print("Modelado omitido porque STAGE =", STAGE)


In [ ]:
#@title 7. Auditoría automática del resultado
import json
import pandas as pd

from urban_denue.config import paths
from urban_denue.constants import ARCHIVE_SHA256, ARCHIVE_SIZE
from urban_denue.archive import verify_archive, sha256_file

p = paths(DENUE_ROOT)
checks = []

def record(name, ok, detail=""):
    checks.append({"check": name, "ok": bool(ok), "detail": str(detail)})
    status = "✓" if ok else "✗"
    print(f"{status} {name}: {detail}")
    if not ok:
        raise AssertionError(f"Falló control: {name}. {detail}")

verify_archive(p.archive)
record("ZIP tamaño", p.archive.stat().st_size == ARCHIVE_SIZE, p.archive.stat().st_size)
record("ZIP SHA-256", sha256_file(p.archive) == ARCHIVE_SHA256, sha256_file(p.archive))

validation_path = p.reports / "validation.json"
if validation_path.exists():
    validation = json.loads(validation_path.read_text(encoding="utf-8"))
    record("25 ediciones", validation.get("editions") == 25, validation.get("editions"))
    record("validación sin errores", validation.get("error_count") == 0, validation.get("error_count"))
    record(
        "observaciones ponderadas históricas",
        validation.get("establishments") == 2_889_999,
        validation.get("establishments"),
    )
else:
    print("ℹ validation.json aún no existe porque STAGE =", STAGE)

state_path = p.processed / "eda_core" / "state_timeseries.parquet"
mun_path = p.processed / "eda_core" / "municipality_timeseries.parquet"

if state_path.exists() and mun_path.exists():
    state = pd.read_parquet(state_path).sort_values("edition_date")
    municipalities = pd.read_parquet(mun_path).sort_values(["municipality_code", "edition_date"])

    record("25 cortes en EDA estatal", len(state) == 25, len(state))

    state_2010 = int(state.loc[state["canonical_edition"].eq("2010-07"), "establishments"].iloc[0])
    state_2026 = int(state.loc[state["canonical_edition"].eq("2026-05"), "establishments"].iloc[0])
    record("Sinaloa 2010", state_2010 == 94_961, state_2010)
    record("Sinaloa 2026-05", state_2026 == 138_882, state_2026)

    maz = municipalities[
        municipalities["municipality_code"].astype(str).str.zfill(3).eq("012")
    ]
    maz_2026 = int(
        maz.loc[maz["canonical_edition"].eq("2026-05"), "establishments"].iloc[0]
    )
    record("Mazatlán 2026-05", maz_2026 == 27_487, maz_2026)
else:
    print("ℹ EDA aún no existe porque STAGE =", STAGE)

audit_df = pd.DataFrame(checks)
display(audit_df)

print("\nAUDITORÍA FINAL: OK")


In [ ]:
#@title 8. Vista rápida del EDA
import pandas as pd
import matplotlib.pyplot as plt

from urban_denue.config import paths

p = paths(DENUE_ROOT)
state_path = p.processed / "eda_core" / "state_timeseries.parquet"
mun_path = p.processed / "eda_core" / "municipality_timeseries.parquet"

if not (state_path.exists() and mun_path.exists()):
    print('Para esta sección usa STAGE = "eda" o una etapa posterior.')
else:
    state = pd.read_parquet(state_path).sort_values("edition_date")
    municipalities = pd.read_parquet(mun_path).sort_values(["municipality_code", "edition_date"])

    print("Serie estatal")
    display(
        state[
            [
                "canonical_edition",
                "establishments",
                "growth_pct",
                "rebenchmark",
                "active_agebs",
                "active_grids",
                "effective_sectors",
            ]
        ]
    )

    mazatlan = municipalities[
        municipalities["municipality_code"].astype(str).str.zfill(3).eq("012")
    ].copy()

    print("\nMazatlán")
    display(
        mazatlan[
            [
                "canonical_edition",
                "establishments",
                "growth_pct",
                "state_share",
                "rank_state",
                "rebenchmark",
            ]
        ]
    )

    ax = mazatlan.plot(
        x="edition_date",
        y="establishments",
        marker="o",
        figsize=(11, 4),
        legend=False,
        title="Mazatlán · establecimientos DENUE por corte",
    )
    ax.set_xlabel("Edición")
    ax.set_ylabel("Establecimientos")
    plt.show()


## Qué hacer después

- Para el EDA normal, deja `STAGE = "eda"`.
- Para métricas espaciales y Moran, usa `STAGE = "spatial"` y activa `WITH_MORAN`.
- Para construir variables econométricas, usa `STAGE = "features"`.
- Para ejecutar todo, incluidos modelos, usa `STAGE = "all"`.

Si `USE_DRIVE_CACHE = True`, el ZIP verificado y los resultados quedan en `MyDrive/Urban_DENUE_Cache`, por lo que las siguientes sesiones de Colab pueden reutilizarlos.